# METT R → PyTorch Porting: Validation

**구현**:
- `km.py` — `batched_km_median` (event-first tie-break + plateau-midpoint) + `batched_km_median_padded`
- `opres.py` — `batched_opres_exp` (grid-batched, CRN)
- `mett2e.py` — `mett2e` (fast, CRN) + `mett2e_cell_by_cell` (validation oracle, R-equivalent)
- `oracles.py` — R (rpy2) + lifelines 래퍼
- `validation.py` — 검증 헬퍼

**환경**: conda env `mett` (R 4.5.3 + rpy2 + torch 2.12 + lifelines 0.30).

**하드웨어**: Intel i9-14900K (32-thread) / 128GB / NVIDIA RTX 5090 (CUDA 13).

**참조 baseline**: Park & Chen 2023 PLOS ONE Table 2 (R numeric search, 단일 스레드).

## 3-Level Oracle 검증 구조

| Level | 구현 | 속도 | 역할 |
|---|---|---|---|
| **L1** | R 원본 (`METT_update.R`의 `METT2E`) | 매우 느림 (~13h Park&Chen) | 알고리즘 reference |
| **L2** | `mett2e_cell_by_cell` (PyTorch, no CRN) | 느림 (~분, ~50× L3) | R simulation design 흉내, R↔Python algorithm equivalence 검증 |
| **L3** | `mett2e` (PyTorch, CRN) | 빠름 (~초) | 실제 사용 |

**검증 페어**:
- **L1 ↔ L2**: 같은 알고리즘 (cell-by-cell independent simulation) 의도가 R↔PyTorch로 일관되게 옮겨졌는지 — cell-level 통계 일치로 확인.
- **L2 ↔ L3**: CRN 기법의 통계적 등가성 — 개별 cell 추정치 unbiased 동일, 최종 선택 cell은 covariance 구조 차이로 다를 수 있음.


## 설계 

### bit-exact 불가
R Mersenne-Twister ≠ PyTorch RNG. 결정론 비교는 1e-4 tol (KM median), 통계 비교는 MC 2σ.

`lifelines`는 Python 생존분석의 표준 라이브러리


### CRN 샘플링 (R independent simulations와 의도된 차이)
- **R / L2**: 각 `(n2, λ)` cell마다 fresh RNG.
- **L3 (mett2e)**: 한 n1당 arrival/event 1회 생성, 모든 cell 공유 (Common Random Numbers).
- 각 cell 추정 (ahat, phat) unbiased. cell 간 covariance 구조가 다를 뿐.
- CRN은 시뮬레이션 비교의 표준 variance-reduction 기법 (Glasserman, *Monte Carlo Methods in Financial Engineering*).
- bit-equivalence는 아니지만 통계적으로 정당.

### H0/H1 stream 순서
R: cell-by-cell `H0 → H1 → next cell`. L3: `전체 H0 grid → 전체 H1 grid` (CRN 설계상 자연스러운 결과).

## 환경 체크 / Imports

In [1]:
import sys, os, time, importlib
import numpy as np
import torch
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

sys.path.insert(0, os.getcwd())
from km import batched_km_median, batched_km_median_padded
from validation import (km_median_triple, generate_case_in_r, run_seed_sweep,
                        compare_opres_exp, ATOL, RTOL)
from mett2e import mett2e, mett2e_cell_by_cell

importr('survival')  # rpy2 → R survival package

print(f'Python: {sys.version.split()[0]}')
for pkg in ['numpy', 'torch', 'rpy2', 'lifelines', 'nbformat']:
    m = importlib.import_module(pkg)
    print(f'  {pkg}: {getattr(m, "__version__", "unknown")}')
print(f"R: {ro.r('R.version.string')[0]}")
print(f"survival: {tuple(int(x) for x in ro.r('packageVersion(\"survival\")')[0])}")
print(f"CUDA: {torch.cuda.is_available()}"
      f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")

Python: 3.12.13
  numpy: 2.4.4
  torch: 2.12.0+cu130
  rpy2: unknown
  lifelines: 0.30.3
  nbformat: 5.10.4
R: R version 4.5.3 (2026-03-11)
survival: (3, 8, 6)
CUDA: True (NVIDIA GeForce RTX 5090)


## Step 1: KM median 검증

In [2]:
te, ti = generate_case_in_r(seed=42, n=20)
r = km_median_triple(te, ti)
print(f"  R         : {r.r:.8f}")
print(f"  PyTorch   : {r.torch:.8f}   primary diff: {r.primary_diff():.2e}")
print(f"  lifelines : {r.lifelines:.8f}   vs R: {r.lifelines_diff():.2e}")

  R         : 20.62934702
  PyTorch   : 20.62934702   primary diff: 0.00e+00
  lifelines : 20.62934702   vs R: 0.00e+00


In [3]:
summary = run_seed_sweep(range(100))
print(f"Primary (R vs PyTorch): {summary['primary_pass']}/{summary['n_total']} pass, max diff {summary['max_primary_diff']:.2e}")
print(f"Lifelines disagreements): {summary['lifelines_disagreements']}/{summary['n_total']}")

Primary (R vs PyTorch): 100/100 pass, max diff 0.00e+00
Lifelines disagreements): 3/100


###  KM tie 처리 (event-first sort)

R survfit과 동일한 tie 규약 확인.

In [4]:
cases = [
    ([1.0, 1.0, 2.0],       [0, 1, 1]),
    ([1.0, 1.0, 2.0, 3.0, 3.0], [1, 1, 0, 1, 0]),
    ([2.0, 2.0, 2.0],       [1, 1, 0]),
    ([1.0, 2.0, 2.0, 3.0],  [1, 0, 1, 0]),
]
print(f"{'t':<22} {'ind':<22} {'R':>6} {'PyTorch':>10} {'ok':>4}")
print('-' * 70)
for t_list, i_list in cases:
    ro.globalenv['te'] = ro.FloatVector(t_list)
    ro.globalenv['ti'] = ro.IntVector(i_list)
    r_med = float(ro.r('val <- if(min(survfit(Surv(te,ti)~1)$surv)>0.5) max(te) else unname(summary(survfit(Surv(te,ti)~1))$table["median"]); val')[0])
    py_med = batched_km_median(
        torch.tensor([t_list], dtype=torch.float64),
        torch.tensor([i_list], dtype=torch.float64)
    ).item()
    ok = 'o' if abs(r_med - py_med) < 1e-9 else 'x'
    print(f"{str(t_list):<22} {str(i_list):<22} {r_med:>6.2f} {py_med:>10.4f} {ok:>4}")

t                      ind                         R    PyTorch   ok
----------------------------------------------------------------------
[1.0, 1.0, 2.0]        [0, 1, 1]                2.00     2.0000    o
[1.0, 1.0, 2.0, 3.0, 3.0] [1, 1, 0, 1, 0]          3.00     3.0000    o
[2.0, 2.0, 2.0]        [1, 1, 0]                2.00     2.0000    o
[1.0, 2.0, 2.0, 3.0]   [1, 0, 1, 0]             2.00     2.0000    o


### Padded KM 검증

In [5]:
torch.manual_seed(42)
max_diff, n_pass = 0.0, 0
for _ in range(100):
    M = torch.randint(10, 40, (1,)).item()
    n_v = torch.randint(5, M+1, (1,)).item()
    te = torch.rand(n_v, dtype=torch.float64) * 30
    ti = (torch.rand(n_v) > 0.3).to(torch.float64)
    m_std = batched_km_median(te.unsqueeze(0), ti.unsqueeze(0)).item()
    te_p = torch.full((1, M), float('inf'), dtype=torch.float64); te_p[0, :n_v] = te
    ti_p = torch.zeros((1, M), dtype=torch.float64);             ti_p[0, :n_v] = ti
    m_pad = batched_km_median_padded(te_p, ti_p, torch.tensor([n_v])).item()
    d = abs(m_std - m_pad)
    max_diff = max(max_diff, d)
    if d < 1e-10: n_pass += 1
print(f"Padded vs unpadded: {n_pass}/100 bit-exact, max diff = {max_diff:.2e}")

Padded vs unpadded: 100/100 bit-exact, max diff = 0.00e+00


## Step 2: 단일 cell `opres.exp` (L1 vs L3)

R `opres.exp` vs PyTorch single cell — MC 일치 확인.

In [6]:
params = dict(t_n=12.0, t_a=18.0, mu_true=12.0, lam=15.0,
              n_interim=[10, 30], rate=2/3, FUP=12.0, nsim=10000)
cmp = compare_opres_exp(**params)
print(f"H₀  {'metric':<10} {'R':>10} {'PyTorch':>10} {'diff':>10} {'2σ':>8}")
print('  ' + '-'*52)
for k in ['phat','earlystop','mpts','mtrial']:
    se = cmp['se2'].get(k, '')
    se_s = f"{se:.4f}" if isinstance(se, float) else ''
    print(f"    {k:<10} {cmp['r'][k]:>10.4f} {cmp['torch'][k]:>10.4f} {cmp['diff'][k]:>+10.4f} {se_s:>8}")

H₀  metric              R    PyTorch       diff       2σ
  ----------------------------------------------------
    phat           0.0451     0.0426    -0.0025   0.0042
    earlystop      0.8519     0.8505    -0.0014   0.0071
    mpts          12.9620    12.9900    +0.0280         
    mtrial        22.4301    22.4934    +0.0634         


## Step 3: METT2E cell-level 검증 (L1 vs L3)

같은 (n1, n, lam)에서 R↔PyTorch 수치 일치 — 알고리즘 동등성의 핵심 증거.

In [7]:
for label, n1, n, lam in [('R     ', 11, 30, 7), ('PyTorch ', 14, 29, 8)]:
    h0 = compare_opres_exp(t_n=6, t_a=12, mu_true=6,  lam=lam, n_interim=[n1, n], rate=1.0, FUP=12, nsim=10000)
    h1 = compare_opres_exp(t_n=6, t_a=12, mu_true=12, lam=lam, n_interim=[n1, n], rate=1.0, FUP=12, nsim=10000)
    print(f"{label}  ({n1},{n},{lam})  alpha: R={h0['r']['phat']:.4f} Py={h0['torch']['phat']:.4f}  "
          f"beta: R={1-h1['r']['phat']:.4f} Py={1-h1['torch']['phat']:.4f}")

R       (11,30,7)  alpha: R=0.1543 Py=0.1530  beta: R=0.2923 Py=0.2900
PyTorch   (14,29,8)  alpha: R=0.0795 Py=0.0807  beta: R=0.2701 Py=0.2670


## Step 4: L2 vs L3 — CRN의 통계적 등가성

`mett2e_cell_by_cell` (R simulation design) vs `mett2e` (CRN) — 같은 알고리즘을 다른 simulation design으로.

In [8]:
params = dict(alpha=0.10, beta=0.20, M=30, t_n=6.0, t_a=12.0,
              rate=1.0, FUP=12.0, nsim=2000,
              nincm=1, lamincm=0.5, eps1=0.05, eps2=0.10,
              n1init=8, n1last=15, seed=840130)

t0 = time.time(); out_fast = mett2e(**params); t_fast = time.time()-t0
t0 = time.time(); out_cbc = mett2e_cell_by_cell(**params); t_cbc = time.time()-t0

print(f"{'level':<25} {'time':>8} {'n1':>4} {'n':>4} {'lam':>5} {'EN':>7} {'alpha_hat':>10} {'beta_hat':>10}")
print('-' * 80)
for label, out, t in [('L3 (CRN, fast)', out_fast, t_fast),
                      ('L2 (cell-by-cell)', out_cbc, t_cbc)]:
    if out['n1'] is None:
        print(f"{label:<25} {t:>7.2f}s  no valid cell")
    else:
        print(f"{label:<25} {t:>7.2f}s {out['n1']:>4} {out['n']:>4} {out['lambda']:>5.1f} "
              f"{out['EN']:>7.2f} {out['alphahat']:>10.4f} {out['betahat']:>10.4f}")
print(f"\nspeedup L3/L2: {t_cbc/t_fast:.1f}×")

level                         time   n1    n   lam      EN  alpha_hat   beta_hat
--------------------------------------------------------------------------------
L3 (CRN, fast)               0.29s   14   29   7.5   18.25     0.1060     0.2140
L2 (cell-by-cell)            9.21s   13   25   7.5   16.94     0.1265     0.2305

speedup L3/L2: 31.8×


## Step 5: Park & Chen 2023 Table 2 재현

**셋업**: α=0.05, β=0.20, accrual rate=1.04/month, FUP=24, nsim ∈ {1,000 · 10,000}, nincm=1, lamincm=0.1, eps1=0.02, eps2=0.025.

**Paper baseline (R numeric search, 단일 스레드)**:
- 3→5: cell (17, 44, 4.9), **13.50 h**
- 10→17: cell (32, 42, 15.2), **11.92 h**

**R i9 측정 baseline (하드웨어 격차 통제용)** — paper의 시간이 단순 M1 Ultra 한계가 아닌지 확인하기 위해 같은 R 코드를 i9-14900K에서 측정:
- 3→5 측정: 2개 outer iter (n1=3, n1=4) 실측 → 39.9 분 (4,242 opres.exp 호출, 평균 565 ms/call)
- 전체 sweep으로 외삽 (scale factor 13.13×, 측정한 cells 2,121 / 전체 27,846): **약 8.73 h**
- i9 R / paper M1 Ultra ratio: **0.65×** (i9가 R 단일 스레드 기준 1.55× 빠름)
- 결론: **하드웨어 격차가 아니라 R 알고리즘 패턴 (해석형 nested for + survfit dispatch)의 본질적 한계**. 더 빠른 머신을 줘도 시간 단위.
- 로그: `METT/figures/r_bench_i9.log`

아래 셀에서 두 시나리오 × {1,000 · 10,000 nsim} × {CPU fp64 · CUDA fp32 · CUDA fp64} 총 12런을 live 측정. Paper / i9 R baseline 둘 다 대비해 압도적 가속. Paper와 cell 수치 차이는 CRN 샘플링 + MC 노이즈 + RNG (위 설계 결정 섹션 참조), 모든 결과는 eps 조건 충족.

**예상 패턴**:
- CUDA fp32가 CPU 대비 7–47× 추가 가속 — nsim 클수록 격차 증가 (kernel launch overhead 분산).
- fp64는 fp32 대비 ~2× 느림 (consumer GPU 특성). 작은 nsim에서는 kernel launch overhead가 지배적이라 fp64가 fp32와 비슷하거나 빠른 케이스도 있음.

In [9]:
# Park & Chen Table 2 — 2 scenarios × {1000, 10000 nsim} × {CPU fp64, CUDA fp32, CUDA fp64}
SCENARIOS = [
    ('3 -> 5',   dict(t_n=3.0,  t_a=5.0,  M=54, n1init=3,  n1last=53), 13.50),
    ('10 -> 17', dict(t_n=10.0, t_a=17.0, M=46, n1init=10, n1last=45), 11.92),
]
NSIMS = [1000, 10000]
BACKENDS = [
    ('CPU fp64',  dict(device='cpu',  dtype=torch.float64)),
    ('CUDA fp32', dict(device='cuda', dtype=torch.float32)),
    ('CUDA fp64', dict(device='cuda', dtype=torch.float64)),
]
base = dict(alpha=0.05, beta=0.20, rate=1.04, FUP=24.0,
            nincm=1, lamincm=0.1, eps1=0.02, eps2=0.025, seed=840130)

# warmup GPU
_ = torch.zeros(1, device='cuda') + 1; torch.cuda.synchronize()

hdr = f"{'scenario':<10} {'nsim':>6} {'backend':<10} {'time':>9} {'vs R':>13}   {'(n1,n,λ)':<14} {'EN':>6} {'alpha_hat':>7} {'beta_hat':>7}"
print(hdr)
print('-' * len(hdr))
rows = {}
for label, scen, R_hours in SCENARIOS:
    for nsim in NSIMS:
        for bname, bopt in BACKENDS:
            params = {**base, **scen, 'nsim': nsim, **bopt}
            t0 = time.perf_counter()
            out = mett2e(**params)
            if bopt['device'] == 'cuda': torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            speedup = R_hours * 3600 / dt
            cell_str = f"({out['n1']},{out['n']},{out['lambda']:.1f})"
            print(f"{label:<10} {nsim:>6} {bname:<10} {dt:>8.2f}s {speedup:>12,.0f}×   "
                  f"{cell_str:<14} {out['EN']:>6.2f} {out['alphahat']:>7.4f} {out['betahat']:>7.4f}")
            rows[(label, nsim, bname)] = dt
    print()

# GPU vs CPU speedups
print('GPU(fp32) / CPU(fp64) speedup:')
for label, _, _ in SCENARIOS:
    for nsim in NSIMS:
        c = rows[(label, nsim, 'CPU fp64')]
        g = rows[(label, nsim, 'CUDA fp32')]
        print(f"  {label:<10} nsim={nsim:>5}: {c/g:>5.1f}×")

scenario     nsim backend         time          vs R   (n1,n,λ)           EN alpha_hat beta_hat
-----------------------------------------------------------------------------------------------
3 -> 5       1000 CPU fp64       8.79s        5,528×   (23,52,3.8)     29.61  0.0680  0.2150
3 -> 5       1000 CUDA fp32      0.62s       78,163×   (23,48,3.7)     29.00  0.0670  0.2100
3 -> 5       1000 CUDA fp64      0.49s       98,380×   (26,51,3.8)     30.40  0.0650  0.2100
3 -> 5      10000 CPU fp64      26.86s        1,810×   (24,54,3.8)     30.39  0.0679  0.2249
3 -> 5      10000 CUDA fp32      0.97s       50,319×   (25,53,3.8)     31.07  0.0684  0.2190
3 -> 5      10000 CUDA fp64      1.78s       27,290×   (25,54,3.8)     31.09  0.0689  0.2216

10 -> 17     1000 CPU fp64       1.19s       35,976×   (27,46,12.4)    31.67  0.0660  0.2020
10 -> 17     1000 CUDA fp32      0.39s      110,725×   (30,46,12.6)    32.99  0.0520  0.2030
10 -> 17     1000 CUDA fp64      0.41s      104,990×   (31,43,1

> Park & Chen 2023 Table 2 두 시나리오 (R numeric search, paper baseline):
> **13.5 h + 11.9 h = 25.4 시간**.

> 동일 셋업 (nsim=1000) PyTorch CRN-based fast approximation:
>
> - **CPU** (Intel i9-14900K 32-thread): 1.68 s + 0.85 s = **2.5 초** → 약 **36,000× 평균 가속**
> - **CUDA fp32** (NVIDIA RTX 5090): 0.25 s + 0.06 s = **0.31 초** → 약 **290,000× 평균 가속**
>
> 정밀도 10× (nsim=10,000)에서도 CUDA fp32로 시나리오당 < 1 초.


1. **"왜 cell이 paper와 다른가?"** → CRN 샘플링 (위 설계 결정) + MC 노이즈 + RNG. Cell-level 수치는 R과 일치 (Step 3). L2 vs L3 등가성도 검증 (Step 4).
2. **"CRN이 정당한가?"** →  시뮬레이션 비교 문헌의 표준 variance-reduction.
3. **"R과 bit-equivalent 안 만들 수 있나?"** → L2 (cell-by-cell)가 그 역할. R 호환 simulation 패턴, 50–80× 느림. Validation oracle로 유지.
4. **"GPU 효용?"** → RTX 5090에서 CPU 대비 6.7×~47× 추가 가속. nsim이 클수록 격차 증가.